# **Cell 1 — Cài thư viện cần thiết**


In [ ]:
!pip install pandas --quiet

#**Cell 2 — Upload file CSV**

In [ ]:
from google.colab import files
import io, pandas as pd

print('Chọn file để upload:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]),
                     header=None, encoding='utf-8', on_bad_lines='skip')
df_raw.columns = ['text']
print(f'Upload thành công: {len(df_raw)} dòng')
df_raw.head()

# **Cell 3 — Làm sạch dữ liệu**

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF\u0300-\u036F.,!?]', ' ', text)
    tokens = text.split()
    meaningful = [t for t in tokens if len(re.sub(r'[^a-zA-Z\u00C0-\u024F\u1E00-\u1EFF]', '', t)) >= 1]
    return re.sub(r'\s+', ' ', ' '.join(meaningful)).strip()

def is_noise(text):
    if not isinstance(text, str) or len(text.strip()) < 3:
        return True
    stripped = text.strip()
    meaningful_chars = re.findall(r'[a-zA-Z\u00C0-\u024F\u1E00-\u1EFF]', stripped)
    if len(stripped) > 0 and len(meaningful_chars) / len(stripped) < 0.3:
        return True
    tokens = stripped.split()
    single_char_ratio = sum(1 for t in tokens if len(t) == 1) / max(len(tokens), 1)
    if single_char_ratio > 0.7 and len(tokens) >= 4:
        return True
    return False

df_raw['text_clean'] = df_raw['text'].apply(clean_text)
df_raw['is_noise']   = df_raw['text_clean'].apply(is_noise)

print(f'Dòng nhiễu   : {df_raw["is_noise"].sum()}')
print(f'Dòng trùng   : {df_raw["text_clean"].duplicated().sum()}')

df_clean = (df_raw[~df_raw['is_noise']]
            .drop_duplicates(subset='text_clean')
            .reset_index(drop=True))
print(f'Sau làm sạch : {len(df_clean)} dòng')

# **Cell 4 — Phân loại Intent**

In [ ]:
INTENT_RULES = {
    'khen_chat_luong':     ['chất_lượng','chất lượng','tốt','tuyệt','ổn','oke','ok',
                            'chắc','bền','đẹp','xinh','xịn','ngon','chuẩn','mềm','nhẹ','êm','siu'],
    'phan_nan_chat_luong': ['kém','tệ','lỗi','hỏng','xấu','không đúng','thất_vọng',
                            'không ổn','không như hình','mùi','nồng','phai','sai màu','sai size'],
    'khen_giao_hang':      ['giao hàng','giao_hàng','shipper','ship','nhanh','đúng hẹn',
                            'đóng gói','đóng_gói','bao bì','nguyên_vẹn'],
    'phan_nan_giao_hang':  ['giao chậm','giao_hàng chậm','lâu','trễ','chưa nhận',
                            'mất hàng','thiếu hàng','sai hàng'],
    'khen_dich_vu':        ['phục_vụ','phục vụ','nhiệt_tình','thân_thiện','tư_vấn',
                            'hỗ_trợ','cảm_ơn','cảm ơn','thank','thanks','đáng_yêu','vui_tính'],
    'phan_nan_dich_vu':    ['thái độ','thái_độ','bất_lịch','không hỗ_trợ',
                            'không phản_hồi','không giải_quyết','lừa đảo'],
    'dung_voi_mo_ta':      ['đúng mô_tả','đúng với mô_tả','giống hình','như hình',
                            'đúng size','đúng màu','chuẩn mô_tả'],
    'de_xuat_mua':         ['nên mua','recommend','rcm','re com men','giới_thiệu',
                            'quay lại','mua tiếp','sẽ mua','đặt tiếp'],
    'gia_tri_gia':         ['giá','rẻ','hợp_lí','hợp lý','đáng tiền','tiết_kiệm',
                            'đáng mua','giá tốt','giá ổn','phải_chăng'],
    'trung_tinh':          [],
}

def classify_intent(text):
    if not isinstance(text, str):
        return 'trung_tinh'
    text_lower = text.lower()
    matched = []
    for intent, keywords in INTENT_RULES.items():
        if intent == 'trung_tinh':
            continue
        for kw in keywords:
            if kw in text_lower:
                matched.append(intent)
                break
    if not matched:
        return 'trung_tinh'
    negative = [i for i in matched if 'phan_nan' in i]
    return negative[0] if negative else matched[0]

df_clean['intent'] = df_clean['text_clean'].apply(classify_intent)

print('Phân bố intent:')
print(df_clean['intent'].value_counts())

# **Cell 5 — Tải xuống file đã làm sạch**

In [ ]:
from google.colab import files

output_clean = 'data_cleaned_with_intent.csv'
df_clean[['text', 'text_clean', 'intent']].to_csv(output_clean,
    index=False, encoding='utf-8-sig')

print(f'Đã lưu: {output_clean} ({len(df_clean)} dòng)')
files.download(output_clean)